### **Objectif**
Extraction automatique des **valeurs uniques** des variables **catégorielles** du dataset.  
Cela permet de :  
- Remplir les menus déroulants dans **Streamlit**  
- Garantir la **cohérence** avec les valeurs vues par le modèle  
- Générer un **JSON de référence** pour l’API (route `/features/choices`)

Le code parcourt chaque variable catégorielle, nettoie les valeurs (`NaN`, espaces, doublons)  
et crée un dictionnaire `value_lists` prêt à être exporté.


### 01. Chargement du dataset

In [1]:
import pandas as pd
import json
from pathlib import Path

df_path = Path("../data/Prepared/prepared_dpe_regroupe_final.csv")
df = pd.read_csv(df_path)
print(f" Dataset chargé : {df.shape[0]} lignes, {df.shape[1]} colonnes")

 Dataset chargé : 358302 lignes, 27 colonnes


### 02. Sélection des colonnes catégorielles

In [2]:
categorical_columns = [
    "isolation_toiture",
    "qualite_isolation_murs",
    "qualite_isolation_menuiseries",
    "type_energie_principale_chauffage",
    "energie_regroupee",
    "type_logement_source",
    "classe_annee_construction"
]

In [4]:
categorical_columns = [c for c in categorical_columns if c in df.columns]
print(f"Colonnes catégorielles retenues : {categorical_columns}")

Colonnes catégorielles retenues : ['isolation_toiture', 'qualite_isolation_murs', 'qualite_isolation_menuiseries', 'type_energie_principale_chauffage', 'energie_regroupee', 'type_logement_source', 'classe_annee_construction']


### 03. Extraction des valeurs uniques

In [6]:
value_lists = {}

for col in categorical_columns:
    uniques = (
        df[col]
        .dropna()
        .astype(str)
        .str.strip()
        .replace({"nan": None})
        .unique()
        .tolist()
    )
    uniques = sorted(list(set(uniques)))
    value_lists[col] = uniques
    print(f"{col}: {len(uniques)} valeurs uniques")


isolation_toiture: 2 valeurs uniques
qualite_isolation_murs: 4 valeurs uniques
qualite_isolation_menuiseries: 4 valeurs uniques
type_energie_principale_chauffage: 12 valeurs uniques
energie_regroupee: 5 valeurs uniques
type_logement_source: 2 valeurs uniques
classe_annee_construction: 6 valeurs uniques


### 04. Sauvegarde en JSON

In [12]:
output_dir = Path("../app/data/value_lists.json")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "value_lists.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(value_lists, f, indent=2, ensure_ascii=False)

print(f"Fichier JSON sauvegardé → {output_file}")

Fichier JSON sauvegardé → ..\app\data\value_lists.json\value_lists.json


In [13]:
with open(output_file, "r", encoding="utf-8") as f:
    data_preview = json.load(f)

print(json.dumps({k: v for k, v in data_preview.items()}, indent=2, ensure_ascii=False))

{
  "isolation_toiture": [
    "0.0",
    "1.0"
  ],
  "qualite_isolation_murs": [
    "BONNE",
    "INSUFFISANTE",
    "MOYENNE",
    "TRÈS BONNE"
  ],
  "qualite_isolation_menuiseries": [
    "BONNE",
    "INSUFFISANTE",
    "MOYENNE",
    "TRÈS BONNE"
  ],
  "type_energie_principale_chauffage": [
    "BOIS – BÛCHES",
    "BOIS – GRANULÉS (PELLETS) OU BRIQUETTES",
    "BOIS – PLAQUETTES D’INDUSTRIE",
    "BOIS – PLAQUETTES FORESTIÈRES",
    "CHARBON",
    "FIOUL DOMESTIQUE",
    "GAZ NATUREL",
    "GPL",
    "PROPANE",
    "RÉSEAU DE CHAUFFAGE URBAIN",
    "ÉLECTRICITÉ",
    "ÉLECTRICITÉ D'ORIGINE RENOUVELABLE UTILISÉE DANS LE BÂTIMENT"
  ],
  "energie_regroupee": [
    "autre",
    "bois",
    "electrique",
    "fioul",
    "gaz"
  ],
  "type_logement_source": [
    "EXISTANT",
    "NEUF"
  ],
  "classe_annee_construction": [
    "1949_1974",
    "1975_1989",
    "1990_1999",
    "2000_2011",
    "apres_2012",
    "avant_1948"
  ]
}
